## Read the data

In [1]:
import numpy as np
import pandas as pd
import sys
PATH_TO_PREDICTOR_HOME = "../.."
sys.path.append(PATH_TO_PREDICTOR_HOME)
METHOD = "hmm_pomegranate"

In [2]:
DATA_PATH_BASE = 'G:/My Drive/Work/ITMO/data'
# DATA_PATH_BASE = 'D:/Google Drive/Work/ITMO/data' #windows
# DATA_PATH_BASE = '/home/denklewer/Data' #ubuntu

DATA_PATH = f'{DATA_PATH_BASE}/length_dependent_hmm/peptide_cores' # work laptop
APPROACH = "DECOY_PATHS"

In [3]:
data_df = pd.read_csv(f"{DATA_PATH}/extractd_cores.csv", index_col=0, sep=';')


In [4]:
ALLELES = list(data_df['allele'].unique())
selected_alleles = ALLELES
models_to_read = ['v']

# Read models

In [13]:
## PARAMETERS
RUNS = 15
# RUNS = 1
TOTAL_SPLITS = 1
SELECTED_RUN = 0
SELECTED_RUN = set(np.arange(0, RUNS))
# SELECTED_RUN = {0}
#
SELECTED_SPLIT = 0
MODELS_DESCRIPTIONS = {
    "viterbi_simple_with_pseudo_and_weights_7": {
        "path": "experiment_results/core_finding_models_viterbi_simple_with_pseudo/",
        "states": 7,
        "params_string": "30-min_iters",
        "combination_size": 1,
        "alleles": ['HLA-DRB1*01:01',
                    # 'HLA-DPA1*01:03-DPB1*04:01'
                    # 'HLA-A*02:01'
                    ],
        "target_lengths": [12, 13, 14, 15, 16, 17, 18],
        "core_start_state": 7,
        "core_end_state": 8

    },
    "viterbi_with_pseudo_and_weights_7": {
        "path": "experiment_results/core_finding_models_viterbi_simple_with_pseudo/",
        "states": 35,
        "params_string": "30-min_iters",
        "combination_size": 1,
        "alleles": ['HLA-DRB1*01:01',
                    # 'HLA-DPA1*01:03-DPB1*04:01'
                    # 'HLA-A*02:01'
                    ],
        "target_lengths": [12, 13, 14, 15, 16, 17, 18],
        "core_start_state": 35,
        "core_end_state": 36,
        "need_to_delete": [37, 38]

    },
    "viterbi_with_pseudo_and_weights_7_reconstructed": {
        "path": "experiment_results/core_finding_models_viterbi_simple_with_pseudo-reconstructed/",
        "states": 35,
        "params_string": "30-min_iters",
        "combination_size": 1,
        "alleles": ['HLA-DRB1*01:01',
                    # 'HLA-DPA1*01:03-DPB1*04:01'
                    # 'HLA-A*02:01'
                    ],
        "target_lengths": [12, 13, 14, 15, 16, 17, 18],
        "core_start_state": 35,
        "core_end_state": 36,
        "need_to_delete": [37, 38]

    },
    "viterbi_with_pseudo_and_weights_7_retrained": {
        "path": "experiment_results/retrained_models/",
        "states": 30,
        "params_string": "30-min_iters",
        "combination_size": 1,
        "alleles": ['HLA-DRB1*01:01',
                    # 'HLA-DPA1*01:03-DPB1*04:01'
                    # 'HLA-A*02:01'
                    ],
        "target_lengths": [9],
        "core_start_state": None,
        "core_end_state": None,
        "need_to_delete": []

    },
    "viterbi_with_pseudo_and_weights_7_retrained_smaller_model": {
        "path": "experiment_results/retrained_models/",
        "states": 20,
        "params_string": "30-min_iters",
        "combination_size": 1,
        "alleles": ['HLA-DRB1*01:01',
                    # 'HLA-DPA1*01:03-DPB1*04:01'
                    # 'HLA-A*02:01'
                    ],
        "target_lengths": [9],
        "core_start_state": None,
        "core_end_state": None,
        "need_to_delete": []

    },

    "viterbi_multiple_7_retrained": {
        "path": "experiment_results/retrained_models/",
        "states": 20,
        "params_string": "30-min_iters",
        "combination_size": 2,
        "alleles": ['HLA-DRB1*01:01',
                    'HLA-DPA1*01:03/DPB1*04:01'
                    # 'HLA-A*02:01'
                    ],
        "target_lengths": [9],
        "core_start_state": None,
        "core_end_state": None,
        "need_to_delete": []

    }
}

In [14]:
MODELS_DESCRIPTIONS.keys()

dict_keys(['viterbi_simple_with_pseudo_and_weights_7', 'viterbi_with_pseudo_and_weights_7', 'viterbi_with_pseudo_and_weights_7_reconstructed', 'viterbi_with_pseudo_and_weights_7_retrained', 'viterbi_with_pseudo_and_weights_7_retrained_smaller_model', 'viterbi_multiple_7_retrained'])

In [15]:
selected_models = [
                 #'single_allele_imbalanced',
                 #'multiple_allele_imbalanced',
                    # 'single_allele_balanced',
                   # 'multiple_allele_balanced',
    # "single_allele_balanced_class2",
    #     "single_allele_balanced_class2_joiners",
    # "single_allele_with_cycle_base_model",
    #  "anchor_model_5_single_allele_balanced_data_attempt3",
    'viterbi_multiple_7_retrained'
    # 'multiple_allele_balanced_longer_training'              
]


In [16]:
from hmm_logic_methods import load_model

In [17]:
import itertools

name_to_run = dict()
name_to_split = dict()
name_to_type = dict()
name_to_model_root = dict()
name_to_target_lengths = dict()
selected_binding_models = list()
per_name_models = dict()


def populate_models_infos(model_name, run_num, split_num, model_path, model_type, target_lengths ):
    selected_binding_models.append(model_name)

    name_to_run[model_name] = run_num
    name_to_split[model_name] = split_num
    name_to_model_root[model_name] = model_path
    assert model_type in ["bw", "v", "bwv", "bw-it", "bwv-it", "v-it"] or "mixed" in model_type
    name_to_type[model_name] = model_type
    name_to_target_lengths[model_name] = target_lengths

for model in selected_models:
    model_description = MODELS_DESCRIPTIONS[model]
    combination_size = model_description['combination_size']
    model_alleles = model_description['alleles']
    MODELS_PATH = model_description['path']
    params_string = model_description['params_string']
    states = model_description['states']
    TARGET_LENGTHS = model_description['target_lengths']
    target_length_string = "-".join([str(l) for l in TARGET_LENGTHS])
    MIXED_ALLELES = list()

    for combination in itertools.combinations(sorted(model_alleles), combination_size):
        allele_name = "+".join(combination)
        # allele_name = f"{allele_name1}+{allele_name2}"
        MIXED_ALLELES.append(allele_name)

    for allele_name in MIXED_ALLELES:
        target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_').replace('/', '_')
        for run_num_b in range(RUNS):
            for split_num_b in range(TOTAL_SPLITS):
                #bw
                if 'bw' in models_to_read:
                    bw_model_name = f"{run_num_b}_run-{target_length_string}-{states}-st-baum-welch-alg-{params_string}_model-{target_allele_name}-{split_num_b}"
                    print(bw_model_name)
                    populate_models_infos(bw_model_name, run_num_b,split_num_b, MODELS_PATH, f"bw-it-mixed({target_allele_name})", TARGET_LENGTHS)
                if 'bwv' in models_to_read:
                    #bwv
                    bwv_model_name = f"{run_num_b}_run-{target_length_string}-{states}-st-baum-welch-alg-{params_string}_model-{target_allele_name}-{split_num_b}_plus_viterbi"
                    populate_models_infos(bwv_model_name, run_num_b,split_num_b, MODELS_PATH, f"bwv-it-mixed({target_allele_name})", TARGET_LENGTHS)
                if 'v' in models_to_read:
                    #v
                    v_model_name = f"{run_num_b}_run-{target_length_string}-{states}-st-viterbi-alg-{params_string}_model-{target_allele_name}-{split_num_b}"
                    populate_models_infos(v_model_name, run_num_b,split_num_b, MODELS_PATH, f"v-it-mixed({target_allele_name})", TARGET_LENGTHS)



for model_name in selected_binding_models:
    per_name_models[model_name] =  load_model(name_to_model_root[model_name], model_name)

In [18]:
list(per_name_models.keys())

['0_run-9-20-st-viterbi-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0',
 '1_run-9-20-st-viterbi-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0',
 '2_run-9-20-st-viterbi-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0',
 '3_run-9-20-st-viterbi-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0',
 '4_run-9-20-st-viterbi-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0',
 '5_run-9-20-st-viterbi-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0',
 '6_run-9-20-st-viterbi-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0',
 '7_run-9-20-st-viterbi-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0',
 '8_run-9-20-st-viterbi-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0',
 '9_run-9-20-st-viterbi-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0',
 '10_run-9-20-st-viterbi-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DR

## Select run and split

In [19]:
for model_key in list(per_name_models.keys()):
    if name_to_split[model_key] != SELECTED_SPLIT:
        del per_name_models[model_key]
    elif type(SELECTED_RUN) == set:
        if name_to_run[model_key] not in SELECTED_RUN:
            del per_name_models[model_key]
    elif name_to_run[model_key] != SELECTED_RUN:
        del per_name_models[model_key]
per_name_models = dict(reversed(sorted(per_name_models.items(), key=lambda item: item[0])))

In [20]:
list(per_name_models.keys())

['9_run-9-20-st-viterbi-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0',
 '8_run-9-20-st-viterbi-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0',
 '7_run-9-20-st-viterbi-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0',
 '6_run-9-20-st-viterbi-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0',
 '5_run-9-20-st-viterbi-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0',
 '4_run-9-20-st-viterbi-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0',
 '3_run-9-20-st-viterbi-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0',
 '2_run-9-20-st-viterbi-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0',
 '1_run-9-20-st-viterbi-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0',
 '14_run-9-20-st-viterbi-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0',
 '13_run-9-20-st-viterbi-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_D

In [21]:
import importlib
import hmm_logic_methods
importlib.reload(hmm_logic_methods)

<module 'hmm_logic_methods' from 'C:\\Projects\\MHC_predictor\\experiments\\length_dependent_hmm\\hmm_logic_methods.py'>

## Extract paths 
We have two approaches: all possible paths and decoy paths

* precision used to crop transitions
* min and max path lengths are  min/max of (TARGET_LENGTHS)

In [22]:
from hmm_logic_methods import get_all_paths_with_probs, get_model_representation

# code for saving paths do model folder
def save_paths_df(paths_df, model, file_name, save_index=False):
    print(f"model {model.name}")
    
    TARGET_PATH_TO_FILES = f"{name_to_model_root[model.name]}{model.name}/"
    paths_df.to_csv("{}/{}".format(TARGET_PATH_TO_FILES,file_name),sep=";", index=save_index)

In [23]:
# exctract paths
if APPROACH == 'total_paths':
    for model_key, model in per_name_models.items():
        print(model_key)
        TARGET_LENGTHS = name_to_target_lengths[model_key]
        paths_df , edge_to_prob = get_all_paths_with_probs(model, cutoff=max(TARGET_LENGTHS), precision=3, min_path_length=min(TARGET_LENGTHS), max_path_length=max(TARGET_LENGTHS))
        save_paths_df(paths_df, model,"total_paths_df.csv")
        break

# Extract only viterbi paths for decoy for models
* we need some data here

## Preparation: get decoy data

In [24]:
TARGET_LENGTHS = list(data_df.core_peptide.str.len().unique())
TARGET_LENGTHS

[9, 10]

In [25]:
from mhcgnomes import parse
# decoy data
def defineOrganism(allele_name):
    parse_result = parse(allele_name)
    if parse_result.is_mouse:
        return "Mouse"
    elif parse_result.is_human:
        return "Human"
    else:
        print("organism not found")
        return None

def defineClass(allele_name):
    parse_result = parse(allele_name)
    if parse_result.has_mhc_class:
        if parse_result.is_class1:
            return 'I'
        elif parse_result.is_class2:
            print('Class II data')
            return 'II'
        else:
            print("Not class I or class II???!")
    else:
        print("Result without class")
    return None

per_allele_decoy_peptides = dict()
for allele_name in ALLELES:
    organism =  defineOrganism(allele_name).lower()
    class_to_use = defineClass(allele_name)
    ## Hard coded here
    class_to_use = 'I'
    DECOY_DATA_PATH = f"{DATA_PATH_BASE}\decoy\sampled_decoy_data_{organism}_class_{class_to_use}.csv"
    decoy_df =  pd.read_csv(DECOY_DATA_PATH)
    decoy_df['length'] = decoy_df['peptide'].str.len()
    per_allele_decoy_peptides[allele_name] = decoy_df.copy()[decoy_df['length'].isin(TARGET_LENGTHS)]['peptide'].values

peptides_for_analysis = np.unique(np.concatenate([value for key, value in per_allele_decoy_peptides.items()
                                                  if key in selected_alleles]))

Class II data
Class II data


## Calculate viterbi paths

In [26]:
decoy_df['length'].value_counts()

9     401984
8     401830
10    400703
11    398606
12    396877
Name: length, dtype: int64

In [27]:
sample_model = per_name_models[list(per_name_models.keys())[0]]
peptide = peptides_for_analysis[0]
logp, path = sample_model.viterbi(peptide)
print("Viterbi for Sequence: '{}'  -- Log Probability: {} -- Path: {}".format(''.join( peptide ), logp, " ".join( state.name for idx, state in path[1:-1] ) ))

Viterbi for Sequence: 'AAAAAAAAA'  -- Log Probability: -28.12169078244688 -- Path: s003 s009 s013 s017 s019 s017 s019 s017 s019


In [28]:
per_name_models.keys()

dict_keys(['9_run-9-20-st-viterbi-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0', '8_run-9-20-st-viterbi-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0', '7_run-9-20-st-viterbi-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0', '6_run-9-20-st-viterbi-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0', '5_run-9-20-st-viterbi-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0', '4_run-9-20-st-viterbi-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0', '3_run-9-20-st-viterbi-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0', '2_run-9-20-st-viterbi-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0', '1_run-9-20-st-viterbi-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0', '14_run-9-20-st-viterbi-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0', '13_run-9-20-st-viterbi-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_D

In [29]:
%%time
from hmm_logic_methods import get_all_viterbi_paths_for_model_and_sequences_mark_core

# exctract paths
for model_key, model in per_name_models.items():
    print(model_key)
    TARGET_LENGTHS = name_to_target_lengths[model_key]
    paths_df = get_all_viterbi_paths_for_model_and_sequences_mark_core(model, peptides_for_analysis, per_model_important_states=None)
    save_paths_df(paths_df, model,"viterbi_paths_df.csv")


# df_bw_paths = get_all_viterbi_paths_for_model_and_sequences(model_bw, peptides_for_analysis)
# df_bwv_paths = get_all_viterbi_paths_for_model_and_sequences(model_bwv, peptides_for_analysis)
# df_v_paths = get_all_viterbi_paths_for_model_and_sequences(model_v, peptides_for_analysis)

9_run-9-20-st-viterbi-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0
Model 9_run-9-20-st-viterbi-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0
Get viterbi paths for each peptide...


 40%|████      | 322997/802687 [00:18<00:27, 17750.18it/s]


KeyboardInterrupt: 

In [22]:
paths_df

,path,norm_tr_prob,len,norm_viterbi_prob,norm_bw_prob,peptide_cnt
0,grouped_model-start>s003>s009>s008>s013>s015>s...,-0.925223,9,-3.817268,-3.194434,569
1,grouped_model-start>s003>s009>s008>s013>s015>s...,-1.628778,10,-4.072638,-3.331562,15
2,grouped_model-start>s003>s009>s008>s013>s015>s...,-1.592434,10,-3.908567,-3.197942,23
3,grouped_model-start>s003>s009>s008>s013>s015>s...,-0.965605,9,-3.847763,-3.207080,410
4,grouped_model-start>s003>s009>s008>s013>s015>s...,-1.699661,10,-4.142732,-3.398359,11
...,...,...,...,...,...,...
19273,grouped_model-start>s001>s002>s009>s008>s013>s...,-1.565846,10,-4.047104,-3.346977,1
19274,grouped_model-start>s001>s002>s009>s008>s013>s...,-1.610038,10,-4.226971,-3.442208,1
19275,grouped_model-start>s001>s002>s009>s008>s012>s...,-1.492445,10,-4.085344,-3.376825,1
19276,grouped_model-start>s004>s001>s000>s008>s013>s...,-1.508872,10,-3.993592,-3.277904,1


# Same thing but extract paths for training data

In [25]:
for selected_allele in model_alleles:
    total_peptides = np.concatenate([per_allele_per_kfold_per_length_binders_train[selected_allele][0][target_length] for target_length in TARGET_LENGTHS])
    print(len(total_peptides), selected_allele)

12001 HLA-DRB1*01:01
8924 HLA-DPA1*01:03-DPB1*04:01


In [21]:
total_peptides = np.concatenate([per_allele_per_kfold_per_length_binders_train[allele_name][0][target_length] for target_length in TARGET_LENGTHS for allele_name in model_alleles])

In [22]:
from hmm_logic_methods import get_all_viterbi_paths_for_model_and_sequences_mark_core

# exctract paths
for model_key, model in per_name_binding_model.items():
    print(model_key)
    TARGET_LENGTHS = name_to_target_lengths[model_key]
    paths_df = get_all_viterbi_paths_for_model_and_sequences_mark_core(model, total_peptides, per_model_important_states, unique_paths=False)
    #    paths_df = get_all_viterbi_paths_for_model_and_sequences(model, peptides_for_analysis)

    save_paths_df(paths_df, model,"viterbi_training_paths_df.csv", save_index=True)

4_run-13-14-15-16-17-18-25-st-baum-welch-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0
Model 4_run-13-14-15-16-17-18-25-st-baum-welch-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0
Get viterbi paths for each peptide...


100%|██████████| 20925/20925 [00:03<00:00, 6475.34it/s]


Subselect unique paths (if unique path argument passed)..
	Unique paths: 20249 


100%|██████████| 20249/20249 [00:01<00:00, 12497.54it/s]


model 4_run-13-14-15-16-17-18-25-st-baum-welch-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0
3_run-13-14-15-16-17-18-25-st-baum-welch-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0
Model 3_run-13-14-15-16-17-18-25-st-baum-welch-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0
Get viterbi paths for each peptide...


100%|██████████| 20925/20925 [00:03<00:00, 6019.41it/s]


... (11 outputs left out)

100%|██████████| 20925/20925 [00:03<00:00, 6321.20it/s]


Subselect unique paths (if unique path argument passed)..
	Unique paths: 20249 


100%|██████████| 20249/20249 [00:01<00:00, 11987.28it/s]


model 0_run-13-14-15-16-17-18-25-st-baum-welch-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0


In [23]:
paths_df

,path,simplified_path,core_start,core_end,norm_tr_prob,len,norm_viterbi_prob,norm_bw_prob,peptide_cnt
RSTYNHLSSWLTD,grouped_model-start>s027>s025>s003>s006>s008>s...,"[0.0, 0.0, 0.0, 1.32, 0.0, 0.0, 1.32, 0.0, 1.3...",3,8,-0.791855,13,-3.368854,-3.047072,1
KMYFALIDTKAYK,grouped_model-start>s027>s025>s003>s006>s008>s...,"[0.0, 0.0, 0.0, 1.27, 0.0, 0.0, 1.27, 0.0, 1.2...",3,8,-0.612463,13,-3.249273,-2.982877,1
PGLARQAPKPRKQ,grouped_model-start>s025>s001>s002>s005>s011>s...,"[0.0, 0.0, 1.35, 0.0, 0.0, 1.35, 0.0, 1.35, 0....",2,7,-0.843387,13,-3.493799,-3.140942,1
AEILELAGNAARD,grouped_model-start>s027>s025>s002>s005>s011>s...,"[0.0, 0.0, 1.25, 0.0, 0.0, 1.25, 0.0, 1.25, 0....",2,7,-0.715628,13,-2.989283,-2.737880,1
SYVSLKAPLTKPL,grouped_model-start>s025>s002>s005>s011>s012>s...,"[0.0, 1.29, 0.0, 0.0, 1.29, 0.0, 1.29, 0.0, 0....",1,6,-1.020668,13,-3.196945,-2.905789,1
...,...,...,...,...,...,...,...,...,...
GIHPTIISESFQKALEKG,grouped_model-start>s027>s025>s001>s004>s000>s...,"[0.0, 0.0, 0.0, 0.0, 0.0, 1.19, 0.0, 0.0, 1.19...",5,10,-0.587096,18,-2.890393,-2.699106,1
PREDRATWKSNYFLKIIQ,grouped_model-start>s027>s025>s001>s004>s001>s...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.35999999...",7,12,-0.845403,18,-3.338233,-2.978306,1
ISEKHDVVFLITKYGYIH,grouped_model-start>s027>s025>s001>s004>s001>s...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.359...",8,13,-0.861269,18,-3.437878,-3.073706,1
LPEPPPLASNLFAQLAEN,grouped_model-start>s027>s025>s001>s004>s001>s...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.29, 0.0, 0.0,...",6,11,-0.774668,18,-3.050593,-2.761140,1


# Same thing but generate random paths from model

In [30]:
data_df

,allele,simplified_path,peptide,core_start,core_end,core_len,peptide_len,core_peptide,core_path
HPSQRLESVSFNQTGPYN,HLA-DPA1*01:03/DPB1*04:01,[ 0. 0. 0. 0. 6. 15.02 15.02 15....,HPSQRLESVSFNQTGPYN,5,13,9,18,LESVSFNQT,[15.02 15.02 15.02 15.02 15.02 15.02 15.02 15....
FEAAVYTLAKTPLIADV,HLA-DRB1*01:01,[ 0. 0. 1.04 1.04 6.2 15.31 15.31 15....,FEAAVYTLAKTPLIADV,5,13,9,17,YTLAKTPLI,[15.31 15.31 15.31 15.31 15.31 15.31 14.27 14....
PDTKKDEAIEIGLK,HLA-DRB1*01:01,[ 0. 4.18 15.33 15.33 15.33 15.33 15.33 15....,PDTKKDEAIEIGLK,2,10,9,14,TKKDEAIEI,[15.33 15.33 15.33 15.33 15.33 15.33 15.33 15....
TGQEMNVAQFLMHMG,HLA-DPA1*01:03/DPB1*04:01,[ 0. 0. 0. 6. 15.04 15.04 15.04 15....,TGQEMNVAQFLMHMG,4,12,9,15,MNVAQFLMH,[15.04 15.04 15.04 15.04 15.04 15.04 15.04 15....
LAPLASSRFLHDHQA,HLA-DPA1*01:03/DPB1*04:01,[ 0. 0. 6.01 15.06 15.06 15.06 15.06 15....,LAPLASSRFLHDHQA,3,11,9,15,LASSRFLHD,[15.06 15.06 15.06 15.06 15.06 15.06 15.06 15....
...,...,...,...,...,...,...,...,...,...
SGHWQSIDFKQV,HLA-DPA1*01:03/DPB1*04:01,[ 0. 0. 7.04 15.04 15.04 15.04 15.04 15....,SGHWQSIDFKQV,3,11,9,12,WQSIDFKQV,[15.04 15.04 15.04 15.04 15.04 15.04 15.04 15....
NTDPYQLMNAVNTLDR,HLA-DRB1*01:01,[ 0. 0. 0. 6.05 15.06 15.06 15.06 15....,NTDPYQLMNAVNTLDR,4,12,9,16,YQLMNAVNT,[15.06 15.06 15.06 15.06 15.06 15.06 15.06 15....
APIIGEYAGEVLRFVGG,HLA-DPA1*01:03/DPB1*04:01,[ 0. 6.1 15.31 15.31 15.31 15.31 15.31 15....,APIIGEYAGEVLRFVGG,2,10,9,17,IIGEYAGEV,[15.31 15.31 15.31 15.31 15.31 15.31 15.31 15....
RVEYHFLSPYVSPKESP,HLA-DRB1*01:01,[ 0. 0. 6.01 15.01 15.01 15.01 15.01 15....,RVEYHFLSPYVSPKESP,3,11,9,17,YHFLSPYVS,[15.01 15.01 15.01 15.01 15.01 15.01 15.01 15....


In [31]:
from hmm_logic_methods import get_all_viterbi_paths_for_model_and_sequences_mark_core

# exctract paths
for model_key, model in per_name_models.items():
    print(model_key)
    TARGET_LENGTHS = name_to_target_lengths[model_key]
    paths_df = get_all_viterbi_paths_for_model_and_sequences_mark_core(model, None, per_model_important_states=None, unique_paths=True, generate_peptides=True)
    #    paths_df = get_all_viterbi_paths_for_model_and_sequences(model, peptides_for_analysis)

    save_paths_df(paths_df, model,"random_paths_df.csv", save_index=True)


9_run-9-20-st-viterbi-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0
Model 9_run-9-20-st-viterbi-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0
Get viterbi paths for each peptide...


  0%|          | 0/50000 [00:00<?, ?it/s]C:\Projects\MHC_predictor\experiments\length_dependent_hmm\hmm_logic_methods.py:1682: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  peptide_arr, path = model.sample(path=True)
100%|██████████| 50000/50000 [00:17<00:00, 2937.94it/s]


Subselect unique paths (if unique path argument passed)..
	Unique paths: 29975 


100%|██████████| 29975/29975 [00:00<00:00, 103705.00it/s]


model 9_run-9-20-st-viterbi-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0
8_run-9-20-st-viterbi-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0
Model 8_run-9-20-st-viterbi-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0
Get viterbi paths for each peptide...


100%|██████████| 50000/50000 [00:18<00:00, 2712.47it/s]


... (51 outputs left out)

100%|██████████| 50000/50000 [00:20<00:00, 2456.22it/s]


Subselect unique paths (if unique path argument passed)..
	Unique paths: 23659 


100%|██████████| 23659/23659 [00:00<00:00, 33964.20it/s]


model 0_run-9-20-st-viterbi-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0


In [25]:
l =5 